In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers


import os
import sys
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime, date, timedelta


# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.ingester.IngesterClass import Ingester
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta
class SurveySummaryIngester(Ingester):
    def __init__(self, arguments):
        super().__init__(arguments)
        self.table = KPI_SurveySummary

    def update_check(self):
        customer_name = self.customer_info['Name']
        customer_id = self.customer_info['CustomerId']
        customer_db = self.customer_info['DBLocation']
        self.Logger.info(f"Processing customer: {customer_name}")

        #Query the reports from KPI_EmissionSources
        query_kpi_survey_summary = f"""SELECT DISTINCT ReportId FROM KPI_SurveySummary WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')"""
        reports_kpi_survey_summary = Query(query = query_kpi_survey_summary).execute(KPIHub_Conn)
        num_reports_kpi_survey_summary = len(reports_kpi_survey_summary)

        #Query the reports from KPI_ReportSummary
        query_kpi_report = f"""SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}'"""
        reports_kpi_hub = Query(query = query_kpi_report).execute(KPIHub_Conn)
        reports_kpi_hub = reports_kpi_hub.drop(0)
        num_reports_kpi_hub = len(reports_kpi_hub)

        #Check if there are new reports
        reports_into = reports_kpi_hub[~reports_kpi_hub['ReportId'].isin(reports_kpi_survey_summary['ReportId'])]
        reports_deleted = reports_kpi_survey_summary[~reports_kpi_survey_summary['ReportId'].isin(reports_kpi_hub['ReportId'])]

        self.data['reports_into'] = reports_into.copy()
        self.data['reports_deleted'] = reports_deleted.copy()
        self.data['num_reports_kpi_survey_summary'] = num_reports_kpi_survey_summary
        self.data['num_reports_kpi_hub'] = num_reports_kpi_hub

        if (num_reports_kpi_hub > 0):
            self.Logger.info(f"Number of reports in KPI_ReportSummary: {num_reports_kpi_hub}")
            if(num_reports_kpi_survey_summary == 0):
                #No reports in the KPI_EmissionSources
                self.Logger.info("No reports in KPIHub, starting from the beginning")
                self.check_flag = True
            else:
                #Reports in the KPIHub
                self.Logger.info(f"Number of reports in KPI_SurveySummary: {num_reports_kpi_survey_summary}")
                if len(reports_into) > 0 or len(reports_deleted) > 0:
                    self.Logger.info(f"Number of new reports into the KPI_SurveySummary: {len(reports_into)}")
                    self.Logger.info(f"Number of deleted reports in the KPI_ReportSummary: {len(reports_deleted)}")
                    self.check_flag = True
                else:
                    self.Logger.info("No new reports into the KPI_SurveySummary or deleted reports in the KPI_SurveySummary")
                    self.check_flag = False
        else:
            self.Logger.info("No reports in KPI_ReportSummary")
            self.check_flag = False

    def query_data(self):
        if self.check_flag and len(self.data['reports_into']) > 0:
            reports_into = self.data['reports_into'].copy()
            reports_into.db.set_query(get_reports(self.customer_info['Name'], starting_date=self.starting_date, report_id_table = '#TempReport', final_checkbox = True))
            reports_to_query = reports_into.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReport')
            reports_to_query.db.set_query(query_surveys_table(report_table="#TempReports"))
            surveys = reports_to_query.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReports')
            surveys.db.set_query(query_segments_table(survey_table="#TempSurvey"))
            self.Logger.info(f"Reports from KPI_ReportSummary: {len(reports_to_query)}")
            self.Logger.info(f"Surveys from LSDB: {len(surveys)}")
            segments = surveys.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'SurveyId', temp_table_name = '#TempSurvey')
            self.Logger.info(f"Segments from LSDB: {len(segments)}")


            # Set the starting time as a datetime object
            surveys['StartHour'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.hour
            surveys['StartTime'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.time
            surveys['EndHour'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.hour
            surveys['EndTime'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.time
            surveys['StartDay'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.date
            surveys['EndDay'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.date

            # Calculate the duration (in minutes) between StartEpoch and EndEpoch for each survey
            surveys['DurationMinutes'] = (
                surveys['EndEpoch'] - surveys['StartEpoch']
            ) / 60      

            # Prepare the report_gdf for report area lookup
            reports_to_query["geometry"] = gpd.GeoSeries.from_wkt(reports_to_query["ReportArea"])

            report_gdf = gpd.GeoDataFrame(
                reports_to_query,
                geometry="geometry",
                crs="EPSG:4326"
            )
            utm_crs = report_gdf.estimate_utm_crs()
            report_gdf = report_gdf.to_crs(utm_crs)

            segments_gdf = gpd.GeoDataFrame(
                segments,
                geometry=gpd.GeoSeries.from_wkt(segments['Shape']),
                crs="EPSG:4326"
            )
            segments_gdf = segments_gdf.to_crs(utm_crs)
        
            survey_summary = surveys.apply(survey_summary_apply, axis=1)
            outputs =  []
            self.data['reports_gdf'] = report_gdf
            self.data['segments_gdf'] = segments_gdf
            self.data['survey_gdf'] = surveys
            for idx, row in report_gdf.iterrows():
                self.Logger.File.info(f"Processing report {row['ReportId']}, {idx+1} of {len(report_gdf)}")
                surveys_subset = surveys[surveys['ReportId'] == row['ReportId']]['SurveyId']
                segments_subset = segments_gdf[segments_gdf['SurveyId'].isin(surveys_subset)]
                segments_in_report = gpd.overlay(
                    segments_subset,
                    gpd.GeoDataFrame([row], geometry=[row['geometry']], crs=segments_subset.crs),
                    how='intersection',
                    keep_geom_type=False
                )

                if segments_in_report.empty:
                    continue  # skip if intersection is empty
                self.data['segments_in_report'] = segments_in_report
                # Convert StartEpoch to datetime (time only, no date)
                segments_in_report['StartTime'] = pd.to_datetime(segments_in_report['StartEpoch'], unit='s').dt.time
                segments_in_report['StartDate'] = pd.to_datetime(segments_in_report['StartEpoch'], unit='s')
                segments_in_report['DayNight'] = segments_in_report['StartTime'].apply(lambda t: get_day_night(t, SUNRISE_TIME, SUNSET_TIME))
                segments_in_report['ActiveIdle'] = segments_in_report['CarSpeedMedian'].apply(lambda x: set_actie_idle(x, SPEED_THRESHOLD))

                # Group by SurveyId and calculate the aggregated metrics per SurveyId (for surveys associated with this report)
                segment_grouped = segments_in_report.groupby('SurveyId')

                for survey_id, group in segment_grouped:
                    output = {
                        'SurveyId': survey_id,
                        'ReportId': row['ReportId'],
                        'DaySegments': (group['DayNight'] == 'Day').sum(),
                        'NightSegments': (group['DayNight'] == 'Night').sum(),
                        'ActiveSegments': (group['ActiveIdle'] == 'Active').sum(),
                        'IdleSegments': (group['ActiveIdle'] == 'Idle').sum(),
                        'TotalSegments': len(group),
                        'TotalKilometers': group['LengthMeters'].sum() / 1000,
                        'DayKilometers': group.loc[group['DayNight'] == 'Day', 'LengthMeters'].sum() / 1000,
                        'NightKilometers': group.loc[group['DayNight'] == 'Night', 'LengthMeters'].sum() / 1000,
                        'SegmentDurationMinutes': group['DurationSeconds'].sum() / 60,
                        'IdleTimeMinutes': group.loc[group['ActiveIdle'] == 'Idle', 'DurationSeconds'].sum() / 60,
                        'ActiveTimeMinutes': group.loc[group['ActiveIdle'] == 'Active', 'DurationSeconds'].sum() / 60,
                        'AvgSpeedKm': 3.6 * group['CarSpeedMedian'].mean(),
                    }
                    outputs.append(output)
       

            output_df = pd.DataFrame(outputs)
            merged_df = pd.merge(
                survey_summary,
                output_df,
                left_on=["SurveyId", "ReportId"],
                right_on=["SurveyId", "ReportId"],
                how="inner"
            )
            merged_df['SegmentWeight'] = merged_df['TotalSegments'] / merged_df['TotalSegmentsInSurvey']
            merged_df['SurveyDurationMinutes'] = merged_df['SurveyRawDurationMinutes'] * merged_df['SegmentWeight']
            self.data['output'] = merged_df
            self.data['output']['LastUpdated'] = datetime.now()
        
        
    def sanity_check(self):
        super().sanity_check()
        #Get all the reports from the KPI_SurveySummary table
        df_surveys = Query(query = f"SELECT DISTINCT ReportId FROM KPI_SurveySummary WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')").execute(KPIHub_Conn)
        self.Logger.info(f"Total number of unique reports from KPI_SurveySummary: {len(df_surveys)}")

    def push_data(self):
        super().push_data(primary_key = ['SurveyId','ReportId'])
        self.Logger.info(f"Data pushed to {db_path}")


# Define a function to determine if survey is in 'day' or 'night'
def get_day_night(start_time, sunrise, sunset):
    # start_time should be a datetime.time object
    hour = start_time.hour
    if sunrise <= hour < sunset:
        return 'Day'
    else:
        return 'Night'

def set_actie_idle(speed, speed_threshold):
    if speed < speed_threshold:
        return 'Idle'
    else:
        return 'Active'

def segment_summary_apply(df):
    return pd.Series({
        'DaySegments': (df['DayNight'] == 'Day').sum(),
        'NightSegments': (df['DayNight'] == 'Night').sum(),
        'ActiveSegments': (df['ActiveIdle'] == 'Active').sum(),
        'IdleSegments': (df['ActiveIdle'] == 'Idle').sum(),
        'TotalSegments': len(df),
        'TotalKilometers': df['LengthMeters'].sum() / 1000,
        'DayKilometers': df.loc[df['DayNight'] == 'Day', 'LengthMeters'].sum() / 1000,
        'NightKilometers': df.loc[df['DayNight'] == 'Night', 'LengthMeters'].sum() / 1000,
        'SegmentDurationMinutes': df['DurationSeconds'].sum() / 60,
        'IdleTimeMinutes': df.loc[df['ActiveIdle'] == 'Idle', 'DurationSeconds'].sum() / 60,
        'ActiveTimeMinutes': df.loc[df['ActiveIdle'] == 'Active', 'DurationSeconds'].sum() / 60,
        'AvgSpeedKm': 3.6*df['CarSpeedMedian'].mean()

    })

def survey_summary_apply(row):
    return pd.Series({
        'SurveyId': row['SurveyId'],
        'SurveyorUnit': row['SurveyorUnit'],
        'SurveyRawDurationMinutes': row['DurationMinutes'],
        'ReportId': row['ReportId'],
        'StartHour': row['StartHour'],
        'StartTime': row['StartTime'],
        'StartEpoch': row['StartEpoch'],
        'EndTime': row['EndTime'],
        'EndEpoch': row['EndEpoch'],
        'StartDay': row['StartDay'],
        'EndDay': row['EndDay'],
        'LateralRotation': row['LateralRotation'],
        'NumberOfPeaks': row['NumberOfPeaks'],
        'TotalSegmentsInSurvey': row['TotalSegmentsInSurvey']
    })


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
customer_list = get_customer_list(KPIHub_Conn)
arguments = {'conn': KPIHub_Conn}

customer_info = customer_list[customer_list['Name'] == 'Avacon'].iloc[0]
surveyIngester = SurveySummaryIngester(arguments)
surveyIngester.set_customer_info(customer_info)
surveyIngester.update_check()
surveyIngester.query_data()
surveyIngester.push_data() 
surveyIngester.delete_data()
surveyIngester.sanity_check()
